In [2]:
with open('data/refined_module_genes.txt', 'r') as f:
    disease_genes = set([line.strip() for line in f])

In [3]:
len(disease_genes)

2148

In [4]:
# In your proximity notebook
import pickle
with open('data/processed/network_G.pkl', 'rb') as f:
    G = pickle.load(f)
print(f"Network loaded: {G.number_of_nodes()} nodes")

Network loaded: 16201 nodes


In [5]:
import networkx as nx   

In [6]:
# Load your drug targets dictionary
with open('data/drug_targets.pkl', 'rb') as f:
    drug_targets = pickle.load(f)

print(f"Total drugs before filtering: {len(drug_targets)}")

# Filter 1: Remove drugs with no targets in network
drug_targets_filtered = {}
for drug, targets in drug_targets.items():
    targets_in_network = set(targets).intersection(set(G.nodes()))
    if len(targets_in_network) > 0:
        drug_targets_filtered[drug] = targets_in_network

print(f"After removing drugs with no network targets: {len(drug_targets_filtered)}")

# Filter 2: Remove drugs with too many targets (non-specific, likely noise)
# Drugs with >50 targets are usually promiscuous binders or data errors
drug_targets_filtered = {
    drug: targets 
    for drug, targets in drug_targets_filtered.items() 
    if len(targets) <= 50
}

print(f"After removing non-specific drugs (>50 targets): {len(drug_targets_filtered)}")

Total drugs before filtering: 18292
After removing drugs with no network targets: 17445
After removing non-specific drugs (>50 targets): 17374


In [ ]:
from collections import defaultdict

print("Pre-computing shortest paths from disease module...")

precomputed_distances = {}

disease_genes_in_network = [g for g in disease_genes if g in G]

for i, gene in enumerate(disease_genes_in_network):
    lengths = nx.single_source_shortest_path_length(G, gene)
    precomputed_distances[gene] = lengths
    
    if i % 50 == 0:
        print(f"Progress: {i}/{len(disease_genes_in_network)}")

print("Pre-computation complete!")

In [8]:
def calculate_proximity_fast(drug_targets, precomputed_distances):
    """
    Creating a function to calculate proximity score for a single drug using pre-computed distances from the disease module.
    """
    
    # Filter drug targets to only those present in network
    drug_targets_in_network = [t for t in drug_targets if t in G]
    
    if not drug_targets_in_network:
        return None
    
    total_distance = 0
    
    for target in drug_targets_in_network:
        # For this drug target, find its distance to every disease gene and store it in a list
        distances_to_disease = []
        
        for disease_gene, distance_map in precomputed_distances.items():
            # Check if this drug target exists in this disease gene's distance map
            if target in distance_map:
                distances_to_disease.append(distance_map[target])
        
        # Take minimum — nearest disease gene to this drug target
        if distances_to_disease:
            total_distance += min(distances_to_disease)
        else:
            # Target completely unreachable from disease module
            total_distance += 10
    
    # Average across all drug targets
    return total_distance / len(drug_targets_in_network)

In [ ]:
import pandas as pd

print("Calculating proximity scores for all drugs...")
proximity_results = {}

drug_list = list(drug_targets_filtered.items())
total_drugs = len(drug_list)

for i, (drug_name, targets) in enumerate(drug_list):
    score = calculate_proximity_fast(
        targets, 
        precomputed_distances, 
    )
    
    if score is not None:
        proximity_results[drug_name] = score
    
    # Progress update every 100 drugs
    if i % 100 == 0:
        print(f"Progress: {i}/{total_drugs} drugs processed")

print(f"\nDone! Scored {len(proximity_results)} drugs")

# Convert to DataFrame and sort
results_df = pd.DataFrame([
    {"drug": drug, "proximity_score": score}
    for drug, score in proximity_results.items()
])

results_df = results_df.sort_values("proximity_score", ascending=True)
print("\nTop 20 drug candidates:")
print(results_df.head(20))

In [ ]:
results_df[results_df['proximity_score'] > 0]